In [52]:
import yfinance as yf

import math

In [53]:
ALL_GOOG = yf.Ticker("GOOG").history(period='max');

In [54]:
ALL_GOOG.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2004-08-19 00:00:00-04:00,2.467295,2.567467,2.367617,2.475684,897427216,0.0,0.0
2004-08-20 00:00:00-04:00,2.492215,2.691326,2.479632,2.672328,458857488,0.0,0.0
2004-08-23 00:00:00-04:00,2.732529,2.799886,2.690586,2.699221,366857939,0.0,0.0
2004-08-24 00:00:00-04:00,2.744619,2.753501,2.555378,2.587453,306396159,0.0,0.0
2004-08-25 00:00:00-04:00,2.589673,2.664679,2.563026,2.615333,184645512,0.0,0.0


In [55]:
ALL_GOOG.tail()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-10-08 00:00:00-04:00,349.644989,353.529999,343.510010,344.859985,13987000,0.0,0.0
2026-10-09 00:00:00-04:00,348.000000,351.059998,346.880005,347.859985,11142900,0.0,0.0
2026-10-12 00:00:00-04:00,348.339996,352.660004,347.160004,348.970001,11854200,0.0,0.0
2026-10-13 00:00:00-04:00,352.415009,354.500000,347.570007,349.519989,14409500,0.0,0.0
2026-10-14 00:00:00-04:00,351.545013,355.250000,346.660004,347.049988,6659390,0.0,0.0


In [56]:
ALL_GOOG.index

DatetimeIndex(['2004-08-19 00:00:00-04:00', '2004-08-20 00:00:00-04:00',
               '2004-08-23 00:00:00-04:00', '2004-08-24 00:00:00-04:00',
               '2004-08-25 00:00:00-04:00', '2004-08-26 00:00:00-04:00',
               '2004-08-27 00:00:00-04:00', '2004-08-30 00:00:00-04:00',
               '2004-08-31 00:00:00-04:00', '2004-09-01 00:00:00-04:00',
               ...
               '2026-10-01 00:00:00-04:00', '2026-10-02 00:00:00-04:00',
               '2026-10-05 00:00:00-04:00', '2026-10-06 00:00:00-04:00',
               '2026-10-07 00:00:00-04:00', '2026-10-08 00:00:00-04:00',
               '2026-10-09 00:00:00-04:00', '2026-10-12 00:00:00-04:00',
               '2026-10-13 00:00:00-04:00', '2026-10-14 00:00:00-04:00'],
              dtype='datetime64[ns, America/New_York]', name='Date', length=5574, freq=None)

In [57]:
# Trim the data to fit the sample data we got from `backtesting`
selected_idx = (ALL_GOOG.index >= '2004-08-19') & (ALL_GOOG.index <= '2013-03-01')
GOOG = ALL_GOOG[selected_idx]

In [58]:
def Momentum(values, n):
    """
    Bet in the direction of the recent price change
    """
    result = []
    for i, v in enumerate(values):
        if i <= n - 1:
            result.append(float('nan'))
            continue
        mom_direction = math.copysign(1, values[i] - values[i-n])
        result.append(mom_direction)
    return result

In [59]:
from backtesting import Strategy
from backtesting.lib import crossover

class MomMr(Strategy):
    # Lookback
    bars = 1
    is_mom = True
    
    def init(self):
        self.mom = self.I(Momentum, self.data.Close, self.bars)
    
    def next(self):
        # Note to self: The last array value (e.g. mom[-1]) is always the most recent value.
        should_go_long = (self.mom[-1] if self.is_mom else -self.mom[-1]) > 0
        if math.isnan(should_go_long):
            return
        # NOTE: tricky part with `not self.position.is_short`
        # print(self.position, " ", self.position.is_long, " ", self.position.is_short, " ", should_go_long)
        if (not self.position or self.position.is_long) and not should_go_long:
            self.position.close()
            self.sell()
        elif (not self.position or self.position.is_short) and should_go_long:
            self.position.close()
            self.buy()

In [60]:
from backtesting import Backtest

# NOTE: increase cash!
bt = Backtest(GOOG, MomMr, cash=100_000, margin=1, commission=.000, finalize_trades=True) # .002
stats = bt.run()
stats

Start                     2004-08-19 00:00...
End                       2013-03-01 00:00...
Duration                   3116 days 01:00:00
Exposure Time [%]                    99.86034
Equity Final [$]                  96089.46888
Equity Peak [$]                  138141.25781
Return [%]                           -3.91053
Buy & Hold Return [%]               644.33559
Return (Ann.) [%]                    -0.46711
Volatility (Ann.) [%]                 34.4727
CAGR [%]                             -0.46649
Sharpe Ratio                         -0.01355
Sortino Ratio                        -0.02036
Calmar Ratio                         -0.00742
Alpha [%]                           -20.93747
Beta                                  0.02643
Max. Drawdown [%]                   -62.91622
Avg. Drawdown [%]                   -17.34975
Max. Drawdown Duration     1864 days 00:00:00
Avg. Drawdown Duration      345 days 00:00:00
# Trades                                 1083
Win Rate [%]                      

In [61]:
# NOTE: interesting! If it's so bad, can we just do the opposite?
a = bt.plot(superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/bokeh/util/serialization.py:243: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


In [65]:
%%time

stats = bt.optimize(bars=[1,2,3,5,7,10,14,21,30], # NOTE: 7 was best so let's extend!! TODO more
                    is_mom=[True, False],
                    maximize='Equity Final [$]',)
                    #constraint=lambda param: param.n1 < param.n2)
stats

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/backtesting.py:1639: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


CPU times: user 448 ms, sys: 31.2 ms, total: 479 ms
Wall time: 457 ms


Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_513c37bb'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_f0a53a8c'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyError: '/psm_782558e2'
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/resource_tracker.py", line 201, in main
    cache[rtype].remove(name)
KeyErr

Start                     2004-08-19 00:00...
End                       2013-03-01 00:00...
Duration                   3116 days 01:00:00
Exposure Time [%]                    99.25512
Equity Final [$]                 966753.47976
Equity Peak [$]                  977719.99373
Return [%]                          866.75348
Buy & Hold Return [%]                687.9875
Return (Ann.) [%]                    30.51173
Volatility (Ann.) [%]                43.30441
CAGR [%]                             30.46523
Sharpe Ratio                          0.70459
Sortino Ratio                         1.46951
Calmar Ratio                          0.74688
Alpha [%]                           808.58695
Beta                                  0.08455
Max. Drawdown [%]                   -40.85238
Avg. Drawdown [%]                    -5.33443
Max. Drawdown Duration      785 days 00:00:00
Avg. Drawdown Duration       43 days 00:00:00
# Trades                                  204
Win Rate [%]                      

In [66]:
stats._strategy

<Strategy MomMr(bars=14,is_mom=)>

In [67]:
bt.plot(plot_volume=False, plot_pl=False, superimpose=False)

/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:717: UserWarning: found multiple competing values for 'toolbar.active_scroll' property; using the latest value
  fig = gridplot(
/Users/alexei/Library/Python/3.9/lib/python/site-packages/bokeh/util/serialization.py:243: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p5869', ...)